In [1]:
import polars as pl

ruta_solar = "/Users/macbook/ProyectosLocales/PrecioLuz/datos/generacion_solar.csv"
ruta_eolica = "/Users/macbook/ProyectosLocales/PrecioLuz/datos/generacion_eolica.csv"

def limpiar(df):
    return (
        df.select(["datetime", "value"])
        .with_columns(
            pl.col("datetime")
            .str.slice(0, 10)
            .str.strptime(pl.Date, "%Y-%m-%d")
            .alias("date"),
            pl.col("value").cast(pl.Float64)
        )
        .drop("datetime")
        .sort("date")
    )

solar = pl.read_csv(ruta_solar, separator=";")
eolica = pl.read_csv(ruta_eolica, separator=";")

solar = limpiar(solar).rename({"value": "solar"})
eolica = limpiar(eolica).rename({"value": "eolica"})

df = solar.join(eolica, on="date")

df = df.with_columns(
    (pl.col("solar") + pl.col("eolica")).alias("renovable_total")
)

# limpiar
nulos = df.null_count()
dup = df.height - df.unique(subset=["date"]).height

df = df.drop_nulls().unique(subset=["date"]).sort("date")

print("Filas:", df.height)
print("Rango:", df["date"].min(), df["date"].max())

print(df.select([
    pl.col("solar").min().alias("solar_min"),
    pl.col("solar").max().alias("solar_max"),
    pl.col("eolica").min().alias("eolica_min"),
    pl.col("eolica").max().alias("eolica_max"),
    pl.col("renovable_total").mean().alias("media_total")
]))

print("Nulos:", nulos)
print("Duplicados:", dup)

# faltantes
fechas = pl.date_range(
    pl.date(2015,1,1),
    pl.date(2025,12,31),
    "1d",
    eager=True
)

faltan = pl.DataFrame({"date": fechas}).join(df, on="date", how="anti")

print("Faltan:", faltan.height)

# guardar
df.write_csv("/Users/macbook/ProyectosLocales/PrecioLuz/datos/renovables_total.csv")

Filas: 2561
Rango: 2018-12-28 2025-12-31
shape: (1, 5)
┌───────────┬──────────────┬────────────┬──────────────┬─────────────┐
│ solar_min ┆ solar_max    ┆ eolica_min ┆ eolica_max   ┆ media_total │
│ ---       ┆ ---          ┆ ---        ┆ ---          ┆ ---         │
│ f64       ┆ f64          ┆ f64        ┆ f64          ┆ f64         │
╞═══════════╪══════════════╪════════════╪══════════════╪═════════════╡
│ 2.027778  ┆ 10029.829861 ┆ 727.104167 ┆ 18167.541667 ┆ 9984.812099 │
└───────────┴──────────────┴────────────┴──────────────┴─────────────┘
Nulos: shape: (1, 4)
┌───────┬──────┬────────┬─────────────────┐
│ solar ┆ date ┆ eolica ┆ renovable_total │
│ ---   ┆ ---  ┆ ---    ┆ ---             │
│ u32   ┆ u32  ┆ u32    ┆ u32             │
╞═══════╪══════╪════════╪═════════════════╡
│ 0     ┆ 0    ┆ 0      ┆ 0               │
└───────┴──────┴────────┴─────────────────┘
Duplicados: 0
Faltan: 1457
